# SECOM 반도체 공정 센서 데이터 - EDA (탐색적 데이터 분석)

**데이터 한계 및 주의사항**
- 이 노트북은 UCI/Kaggle에 공개된 **익명화(anonymized)된 SECOM 데이터셋**을 다룹니다.
- 590개의 센서 피처(`feature_000` ~ `feature_589`)는 실제 반도체 Fab의 특정 장비·챔버·공정 단계와 매핑되어 있지 않습니다.
- 따라서 본 분석은 **가상 공정 수율 분석 시뮬레이션**이며, 특정 feature가 실제 어떤 공정을 의미한다고 단정하지 않습니다.
- 실제 Fab 데이터의 물리적 인과관계를 규명하는 분석이 아니라, 공개 데이터에서 불량 판별에 통계적으로 기여하는 센서 패턴을 탐색하는 것이 목적입니다.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if (Path.cwd() / 'notebooks').exists() is False and Path.cwd().name == 'notebooks' else Path.cwd()
if (Path.cwd().name == 'notebooks'):
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from src import config, preprocess
from src.data_loader import load_raw_secom, get_feature_columns
from src.utils import ensure_dir, save_json, save_current_figure, set_random_seed

set_random_seed(config.RANDOM_STATE)
sns.set_theme(style='whitegrid')
ensure_dir(config.FIGURES_DIR)
ensure_dir(config.METRICS_DIR)
pd.set_option('display.max_columns', 20)

## 1. 원본 데이터 로드 및 파일 구조 확인
`src/data_loader.py`가 `data/raw/` 아래에서 Kaggle 결합 CSV(`uci-secom.csv`) 또는 UCI 2-파일 포맷(`secom.data` + `secom_labels.data`)을 자동으로 탐지합니다.

In [2]:
df = load_raw_secom(config.DATA_RAW_DIR)
feature_cols = get_feature_columns(df)
print(f'rows={len(df)}, columns={df.shape[1]}, feature columns={len(feature_cols)}')
df.head()

rows=1567, columns=595, feature columns=590


,row_id,Time,Pass_Fail_Raw,Pass_Fail,Pass_Fail_Label,feature_000,feature_001,feature_002,feature_003,feature_004,...,feature_580,feature_581,feature_582,feature_583,feature_584,feature_585,feature_586,feature_587,feature_588,feature_589
0,0,2008-07-19 11:55:00,-1,0,Pass,3030.93,2564.00,2187.7333,1411.1265,1.3602,...,NaN,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN
1,1,2008-07-19 12:32:00,-1,0,Pass,3095.78,2465.14,2230.4222,1463.6606,0.8294,...,0.0060,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045
2,2,2008-07-19 13:17:00,1,1,Fail,2932.61,2559.94,2186.4111,1698.0172,1.5102,...,0.0148,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602
3,3,2008-07-19 14:43:00,-1,0,Pass,2988.72,2479.90,2199.0333,909.7926,1.3204,...,0.0044,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432
4,4,2008-07-19 15:22:00,-1,0,Pass,3032.24,2502.87,2233.3667,1326.5200,1.5334,...,NaN,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432


## 2~4. Feature 컬럼명 / Time / Pass_Fail 병합 확인
`load_raw_secom()`이 이미 `feature_000..feature_589`, `Time`, `Pass_Fail`(0/1), `Pass_Fail_Label`(Pass/Fail)을 하나의 테이블로 병합합니다.

In [3]:
print(feature_cols[:3], '...', feature_cols[-3:])
df[['row_id', config.TIME_COL, config.LABEL_RAW_COL, config.LABEL_COL, config.LABEL_TEXT_COL]].head()

['feature_000', 'feature_001', 'feature_002'] ... ['feature_587', 'feature_588', 'feature_589']


,row_id,Time,Pass_Fail_Raw,Pass_Fail,Pass_Fail_Label
0,0,2008-07-19 11:55:00,-1,0,Pass
1,1,2008-07-19 12:32:00,-1,0,Pass
2,2,2008-07-19 13:17:00,1,1,Fail
3,3,2008-07-19 14:43:00,-1,0,Pass
4,4,2008-07-19 15:22:00,-1,0,Pass


## 5. 행/열 수, 데이터 타입, 중복 행, 라벨 분포

In [4]:
n_rows, n_cols = df.shape
n_duplicates = int(df[feature_cols].duplicated().sum())
label_counts = df[config.LABEL_TEXT_COL].value_counts()
fail_rate = float(df[config.LABEL_COL].mean())

print(f'rows={n_rows}, cols={n_cols}')
print(df.dtypes.value_counts())
print(f'duplicate feature rows={n_duplicates}')
print(label_counts)
print(f'fail rate={fail_rate:.4f}')

rows=1567, cols=595
float64           590
int64               3
datetime64[us]      1
str                 1
Name: count, dtype: int64
duplicate feature rows=0
Pass_Fail_Label
Pass    1463
Fail     104
Name: count, dtype: int64
fail rate=0.0664


In [5]:
fig, ax = plt.subplots(figsize=(5, 4))
label_counts.plot(kind='bar', color=['#4C72B0', '#C44E52'], ax=ax)
ax.set_title('Pass/Fail Class Distribution')
ax.set_xlabel('Class')
ax.set_ylabel('Count')
for i, v in enumerate(label_counts.values):
    ax.text(i, v, str(v), ha='center', va='bottom')
save_current_figure(config.FIGURES_DIR / 'class_distribution.png')
plt.show()

## 6. 결측치 비율 계산 및 시각화

In [6]:
missing_ratio = preprocess.compute_missing_ratio(df, feature_cols)
missing_ratio.describe()

count    590.000000
mean       0.045375
std        0.154340
min        0.000000
25%        0.001276
50%        0.003829
75%        0.005743
max        0.911934
dtype: float64

In [7]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(missing_ratio, bins=40, color='#4C72B0')
ax.set_title('Feature Missing-Value Ratio Histogram')
ax.set_xlabel('Missing Ratio')
ax.set_ylabel('Number of Features')
save_current_figure(config.FIGURES_DIR / 'missing_ratio_histogram.png')
plt.show()

In [8]:
top30_missing = missing_ratio.head(30)
fig, ax = plt.subplots(figsize=(7, 8))
ax.barh(top30_missing.index[::-1], top30_missing.values[::-1], color='#C44E52')
ax.set_title('Top 30 Missing Features')
ax.set_xlabel('Missing Ratio')
save_current_figure(config.FIGURES_DIR / 'top30_missing_features.png')
plt.show()

## 7. 상수 변수 및 저분산 변수 탐지

In [9]:
constant_features = preprocess.detect_constant_or_low_variance_features(df, feature_cols)
print(f'constant/near-zero-variance features: {len(constant_features)} / {len(feature_cols)}')

fig, ax = plt.subplots(figsize=(4, 4))
ax.bar(['Usable', 'Constant/Low-Variance'], [len(feature_cols) - len(constant_features), len(constant_features)],
       color=['#55A868', '#C44E52'])
ax.set_title('Constant / Low-Variance Feature Count')
ax.set_ylabel('Number of Features')
save_current_figure(config.FIGURES_DIR / 'constant_feature_count.png')
plt.show()

constant/near-zero-variance features: 127 / 590


## 8. 결측치 비율 임계값 비교 (40% / 50% / 70%)
임계값에 따라 몇 개의 피처가 제거 대상이 되는지 비교합니다. 실제 모델링 파이프라인은 학습 데이터에서만 이 임계값을 적용합니다 (`src/preprocess.MissingRatioDropper`, `config.MODELING_MISSING_THRESHOLD`).

In [10]:
threshold_comparison = pd.DataFrame({
    'threshold': config.MISSING_RATIO_THRESHOLDS,
    'n_features_dropped': [len(preprocess.features_above_missing_threshold(missing_ratio, t)) for t in config.MISSING_RATIO_THRESHOLDS],
})
threshold_comparison['n_features_kept'] = len(feature_cols) - threshold_comparison['n_features_dropped']
threshold_comparison

,threshold,n_features_dropped,n_features_kept
0,0.4,32,558
1,0.5,28,562
2,0.7,8,582


## 9. 상관관계가 높은 변수쌍 탐색
590개 전체 상관행렬은 시각화가 어려우므로, 분산 상위 feature를 대상으로 계산합니다.

In [11]:
high_corr_pairs = preprocess.find_high_correlation_pairs(df, feature_cols, threshold=config.HIGH_CORR_THRESHOLD)
print(f'|corr| > {config.HIGH_CORR_THRESHOLD} pairs: {len(high_corr_pairs)}')
high_corr_pairs.head(20)

|corr| > 0.95 pairs: 329


,feature_1,feature_2,correlation
0,feature_358,feature_544,1.0
1,feature_342,feature_347,1.0
2,feature_580,feature_588,1.0
3,feature_579,feature_587,1.0
4,feature_578,feature_586,1.0
5,feature_085,feature_542,1.0
6,feature_074,feature_347,1.0
7,feature_492,feature_545,1.0
8,feature_074,feature_209,1.0
9,feature_074,feature_342,1.0


In [12]:
top_var_features = preprocess.top_variance_features(df, feature_cols, n=30)
corr_subset = df[top_var_features].corr()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr_subset, cmap='coolwarm', center=0, square=True, ax=ax,
            cbar_kws={'label': 'Pearson correlation'})
ax.set_title('Correlation Heatmap - Top 30 High-Variance Features')
save_current_figure(config.FIGURES_DIR / 'correlation_heatmap_top_variance.png')
plt.show()

## 10. 정상/불량 간 주요 센서 분포 비교
분산이 큰 상위 feature 중 일부를 대상으로 Pass/Fail 그룹 간 분포를 비교합니다. 이는 단순 시각적 비교이며 통계적 유의성 검정을 대체하지 않습니다.

In [13]:
candidate_sensors = top_var_features[:6]
plot_df = df[candidate_sensors + [config.LABEL_TEXT_COL]].melt(
    id_vars=config.LABEL_TEXT_COL, var_name='feature', value_name='value'
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=plot_df, x='feature', y='value', hue=config.LABEL_TEXT_COL, ax=ax)
ax.set_title('Sensor Distribution by Pass/Fail (Top Variance Candidates)')
plt.xticks(rotation=30)
save_current_figure(config.FIGURES_DIR / 'sensor_distribution_by_class.png')
plt.show()

## 11. PCA 2D 시각화
**주의**: 아래 PCA 시각화는 원본 고차원 데이터를 2차원으로 단순 투영한 결과이며, Pass/Fail이 시각적으로 뚜렷하게 분리되지 않는 것이 정상입니다.
이 그래프를 '모델이 잘 분류할 수 있다는 증거'로 과장 해석하지 않습니다. PCA는 분산이 큰 방향을 보존할 뿐, 클래스 분리를 보장하지 않습니다.

In [14]:
pca_input = df[feature_cols]
imputed = SimpleImputer(strategy='median').fit_transform(pca_input)
scaled = StandardScaler().fit_transform(imputed)
pca = PCA(n_components=2, random_state=config.RANDOM_STATE)
components = pca.fit_transform(scaled)

fig, ax = plt.subplots(figsize=(7, 6))
for label_val, color, name in [(0, '#4C72B0', 'Pass'), (1, '#C44E52', 'Fail')]:
    mask = df[config.LABEL_COL].values == label_val
    ax.scatter(components[mask, 0], components[mask, 1], s=10, alpha=0.5, c=color, label=name)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('PCA 2D Projection (exploratory only, not proof of separability)')
ax.legend()
save_current_figure(config.FIGURES_DIR / 'pca_2d_projection.png')
plt.show()

## Time 구조 확인 (시간 기반 분할 타당성 검토)
`Time`의 순서성과 월별 불량률 추이를 확인하여, 시간 기반(chronological) 분할이 필요한지 검토합니다.
뚜렷한 시간 추세가 없다면 기본 모델 평가는 stratified random split을 사용합니다 (본 프로젝트의 기본 전략).

In [15]:
time_structure = preprocess.analyze_time_structure(df)
print('is_monotonic_time:', time_structure['is_monotonic_time'])
print('time_span_days:', time_structure['time_span_days'])
print('fail_rate_std_across_months:', time_structure['fail_rate_std_across_months'])
pd.Series(time_structure['fail_rate_by_month']).sort_index()

is_monotonic_time: False
time_span_days: 337.0
fail_rate_std_across_months: 0.036724103424010716


2008-01    0.058824
2008-02    0.051020
2008-03    0.020000
2008-04    0.061224
2008-05    0.112903
2008-06    0.089552
2008-07    0.140351
2008-08    0.080679
2008-09    0.041162
2008-10    0.048780
2008-11    0.057143
2008-12    0.000000
dtype: float64

## 12. EDA 요약 저장
핵심 EDA 결과를 `outputs/metrics/eda_summary.json`에 저장합니다.

In [16]:
eda_summary = {
    'n_rows': n_rows,
    'n_columns': n_cols,
    'n_feature_columns': len(feature_cols),
    'n_duplicate_feature_rows': n_duplicates,
    'fail_rate': fail_rate,
    'label_counts': label_counts.to_dict(),
    'missing_ratio_summary': {
        'mean': float(missing_ratio.mean()),
        'median': float(missing_ratio.median()),
        'max': float(missing_ratio.max()),
    },
    'missing_ratio_threshold_comparison': threshold_comparison.to_dict(orient='records'),
    'n_constant_or_low_variance_features': len(constant_features),
    'n_high_correlation_pairs': int(len(high_corr_pairs)),
    'high_correlation_threshold': config.HIGH_CORR_THRESHOLD,
    'time_structure': time_structure,
    'pca_explained_variance_ratio_2d': pca.explained_variance_ratio_.tolist(),
    'notes': (
        'All feature IDs are anonymized SECOM sensor variables from a public dataset. '
        'No physical equipment/chamber/process mapping exists or is assumed in this project.'
    ),
}
save_json(eda_summary, config.METRICS_DIR / 'eda_summary.json')
eda_summary

{'n_rows': 1567,
 'n_columns': 595,
 'n_feature_columns': 590,
 'n_duplicate_feature_rows': 0,
 'fail_rate': 0.06636885768985322,
 'label_counts': {'Pass': 1463, 'Fail': 104},
 'missing_ratio_summary': {'mean': 0.04537548808583822,
  'median': 0.0038289725590299937,
  'max': 0.9119336311423102},
 'missing_ratio_threshold_comparison': [{'threshold': 0.4,
   'n_features_dropped': 32,
   'n_features_kept': 558},
  {'threshold': 0.5, 'n_features_dropped': 28, 'n_features_kept': 562},
  {'threshold': 0.7, 'n_features_dropped': 8, 'n_features_kept': 582}],
 'n_constant_or_low_variance_features': 127,
 'n_high_correlation_pairs': 329,
 'high_correlation_threshold': 0.95,
 'time_structure': {'is_monotonic_time': False,
  'time_span_days': 337.0,
  'fail_rate_by_month': {'2008-01': 0.058823529411764705,
   '2008-02': 0.05102040816326531,
   '2008-03': 0.02,
   '2008-04': 0.061224489795918366,
   '2008-05': 0.11290322580645161,
   '2008-06': 0.08955223880597014,
   '2008-07': 0.14035087719298245